# Bible Corpus: Pairwise Domain Distance

Downloads selected language bibles from https://github.com/christos-c/bible-corpus and evaluates
pairwise domain-distance metrics from `corpus_helpers.metrics` against two external ground truths:
URIEL typological feature distances and a manually specified phylogenetic tree.

**Languages** — chosen to give clear within-family and cross-family signal:
- **West Germanic**: `en` English, `de` German, `nl` Dutch
- **North Germanic**: `da` Danish, `sv` Swedish
- **West Slavic** (Latin script): `cs` Czech, `sk` Slovak, `pl` Polish
- **South Slavic** (Latin script): `hr` Croatian

**Dependency**: `pip install lang2vec` for the URIEL evaluation.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
# !pip install jinja2
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, cophenet
from scipy.spatial.distance import squareform
from scipy.stats import spearmanr
from functools import partial
import pickle
from corpus_helpers.read import (
    lower, 
    split_line, 
    delete_newline, 
    split_word, 
    delete_blank, 
    as_bytes, 
    chain_preprocessors
)

from corpus_helpers.metrics import (
    ngram_overlap,
    ngram_divergence,
    normalized_compression_distance,
    bpe_overlap, 
    sampled_distance
)

/Users/rastislavhronsky/Research/corpus-helpers/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download Bible XML files

In [ ]:
SEED = 7
LANGUAGES = {
    'en': 'English',
    'de': 'German',
    'nl': 'Dutch',
    'da': 'Danish',
    'no': 'Norwegian',
    'sv': 'Swedish',
    'cs': 'Czech',
    'sk': 'Slovak',
    'si': 'Slovene',
    'pl': 'Polish',
    'hr': 'Croatian',
    'rs': 'Serbian',
    'ro': 'Romanian',
    'hu': 'Hungarian',
    'fi': 'Finnish',
}
LANG_CODES = list(LANGUAGES.keys())
N = len(LANG_CODES)

RAW_BASE = 'https://raw.githubusercontent.com/christos-c/bible-corpus/master/bibles'
CACHE_DIR = Path('_bible_cache')
CACHE_DIR.mkdir(exist_ok=True)

def download(lang_code: str, lang_name: str) -> Path:
    dest = CACHE_DIR / f'{lang_code}.xml'
    if not dest.exists():
        url = f'{RAW_BASE}/{lang_name}.xml'
        print(f'Downloading {lang_name}...', end=' ', flush=True)
        urllib.request.urlretrieve(url, dest)
        print('done')
    else:
        print(f'{lang_name}: cached')
    return dest

paths = {code: download(code, name) for code, name in LANGUAGES.items()}

English: cached
German: cached
Dutch: cached
Danish: cached
Norwegian: cached
Swedish: cached
Czech: cached
Slovak: cached
Slovene: cached
Polish: cached
Croatian: cached
Serbian: cached
Romanian: cached
Hungarian: cached
Finnish: cached


## 2. Parse XML → list of verse bytes

In [4]:

PRETOKEN_PIPELINE = [lower, split_line, delete_newline, split_word, delete_blank]

def parse_bible(path: Path) -> list[bytes]:
    tree = ET.parse(path)
        # (seg.text or '').strip().encode('utf-8')
    segments = [(seg.text or '').strip() for seg in tree.iter('seg')
                if seg.text and seg.text.strip()]
    return chain_preprocessors(segments, PRETOKEN_PIPELINE)

paths = {code: download(code, name) for code, name in LANGUAGES.items()}
corpora: dict[str, list[bytes]] = {}
for code, path in paths.items():
    corpora[code] = list(parse_bible(path))
    print(f'{code}: {len(corpora[code]):,} verses')

English: cached
German: cached
Dutch: cached
Danish: cached
Norwegian: cached
Swedish: cached
Czech: cached
Slovak: cached
Slovene: cached
Polish: cached
Croatian: cached
Serbian: cached
Romanian: cached
Hungarian: cached
Finnish: cached
en: 918,445 verses
de: 820,005 verses
nl: 839,790 verses
da: 786,197 verses
no: 841,593 verses
sv: 860,278 verses
cs: 731,091 verses
sk: 746,897 verses
si: 777,701 verses
pl: 741,355 verses
hr: 697,515 verses
rs: 684,520 verses
ro: 905,226 verses
hu: 741,451 verses
fi: 676,152 verses


## 3. Compute pairwise distance matrices

In [12]:
import bz2, lzma

COMPRESSORS = {
    'zlib':  None,                           # default in normalized_compression_distance
    'bz2':   bz2.compress,
    'lzma':  lzma.compress,
}

def ncd_sym(compressor):
    """Return a symmetric NCD callable suitable for sampled_distance."""
    return partial(normalized_compression_distance, compressor=compressor, symmetric=True)

METRICS_NCD = []
for sz in [256, 1024, 4096, 16384, 65536]: 
    for comp_fn in COMPRESSORS: 
        METRICS_NCD.append(dict(
            label=f'ncd_{comp_fn}_{sz}', 
            fn=partial(sampled_distance, metric=ncd_sym(COMPRESSORS[comp_fn]), sample_size_per_iteration=sz, k=10, threshold=.001, max_iterations=100, seed=SEED, )))
        
METRICS_JSD = [
    dict(label=f'jsd_char_n={n}', fn=partial(ngram_divergence, method='jsd', n=n, unit='char'))
    for n in [3,4,5,6,7]
]

METRICS_JACC = [
    dict(label='jacc_word', fn=partial(ngram_overlap, n=1, unit='word')),
]

k_steps = 25
METRICS_BPE = [
    dict(label=f'bpe_overlap_k-{k_steps}_mff-1e-5', fn=partial(bpe_overlap, k_steps=k_steps, log_k_auc=True, return_dict=False, min_freq_fract=1e-5, use_skyline=False)),
    dict(label=f'bpe_overlap_k-{k_steps}_mff-1e-7', fn=partial(bpe_overlap, k_steps=k_steps, log_k_auc=True, return_dict=False, min_freq_fract=1e-7, use_skyline=False)),
    dict(label=f'bpe_overlap_k-{k_steps}_mff-1e-5_sky', fn=partial(bpe_overlap, k_steps=k_steps, log_k_auc=True, return_dict=False, min_freq_fract=1e-5, min_freq_fract_b=1e-8, use_skyline=True)),
    dict(label=f'bpe_overlap_k-{k_steps}_mff-1e-7_sky', fn=partial(bpe_overlap, k_steps=k_steps, log_k_auc=True, return_dict=False, min_freq_fract=1e-7, min_freq_fract_b=1e-8, use_skyline=True)),
]

METRICS = METRICS_NCD + METRICS_JSD + METRICS_BPE + METRICS_JACC

In [ ]:
_path_metric_mats = Path('_bible_cache/mats.pkl')

if _path_metric_mats.exists(): 
    with _path_metric_mats.open('rb') as f: 
        METRIC_MATS = pickle.load(f)

METRIC_MATS = {m['label']: np.zeros((N, N)) for m in METRICS}
pairs = list(combinations(range(N), 2))
print(f'Computing {len(pairs)} pairs × {len(METRICS)} metrics...')

for i, j in pairs:
    ca, cb = corpora[LANG_CODES[i]], corpora[LANG_CODES[j]]
    print(f"  {LANG_CODES[i]}-{LANG_CODES[j]}", end='  ', flush=True)
    for m in METRICS:
        if METRIC_MATS[m['label']][i, j] != np.float64(0): 
            continue
        val = m['fn'](ca, cb)
        METRIC_MATS[m['label']][i, j] = val
        METRIC_MATS[m['label']][j, i] = val
        print(f"{m['label']}={val:.4f}", end='  ', flush=True)
    
    with open('_bible_cache/mats.pkl', 'wb') as f: 
        pickle.dump(METRIC_MATS, f)
    print()

print('Done.')

## 4. URIEL evaluation

[URIEL](http://www.cs.cmu.edu/~dmortens/uriel.html) encodes typological properties of languages as feature vectors.
`lang2vec` provides convenient access. We fetch three feature sets and compute cosine distances:
- **`fam`** — one-hot language family membership (should closely match the gold tree)
- **`phonology_knn`** — phonological features interpolated via k-NN
- **`syntax_knn`** — syntactic features interpolated via k-NN

For each feature set × metric class (NCD, JSD, BPE, Jaccard) we produce a separate scatter PDF
and measure Spearman ρ against the URIEL distances.

In [13]:
# setuptools ≥ 72 dropped pkg_resources; shim it before lang2vec imports it
import sys, types, importlib.util, os as _os
if 'pkg_resources' not in sys.modules:
    _pr = types.ModuleType('pkg_resources')
    def _resource_filename(pkg, path):
        spec = importlib.util.find_spec(pkg)
        return _os.path.join(_os.path.dirname(spec.origin), path)
    _pr.resource_filename = _resource_filename
    sys.modules['pkg_resources'] = _pr

import lang2vec.lang2vec as l2v
from sklearn.metrics.pairwise import cosine_distances

ISO3 = {
    'en': 'eng', 'de': 'deu', 'nl': 'nld', 'da': 'dan', 'no': 'nor',
    'sv': 'swe', 'cs': 'ces', 'sk': 'slk', 'si': 'slv', 'pl': 'pol',
    'hr': 'hrv', 'rs': 'srp', 'ro': 'ron', 'hu': 'hun', 'fi': 'fin',
}
iso3_codes = [ISO3[c] for c in LANG_CODES]

URIEL_SETS = {
    'family':       'fam',
    'phonological': 'phonology_knn',
    'syntax':       'syntax_knn',
}

uriel_mats: dict[str, np.ndarray] = {}
for name, fset in URIEL_SETS.items():
    feats = l2v.get_features(iso3_codes, fset)
    vecs = np.array([feats[c] for c in iso3_codes], dtype=float)
    valid = ~np.any(np.isnan(vecs), axis=0)
    vecs_clean = vecs[:, valid]
    print(f'{name}: {vecs.shape[1]} dims → {valid.sum()} after NaN filter')
    uriel_mats[name] = cosine_distances(vecs_clean)

print()
print('Lowest family distances (should all be within-family):')
fam = uriel_mats['family']
idx = np.triu_indices(N, k=1)
order = np.argsort(fam[idx])
for rank, o in enumerate(order[:5]):
    i, j = idx[0][o], idx[1][o]
    print(f'  {rank+1}. {LANG_CODES[i]}-{LANG_CODES[j]}  d={fam[i,j]:.4f}')

family: 3718 dims → 3718 after NaN filter
phonological: 28 dims → 28 after NaN filter
syntax: 103 dims → 103 after NaN filter

Lowest family distances (should all be within-family):
  1. hr-rs  d=0.0000
  2. cs-sk  d=0.0871
  3. si-rs  d=0.1548
  4. si-hr  d=0.1548
  5. da-sv  d=0.1667


In [49]:
from scipy.stats import pearsonr

_path_metric_mats = Path('_bible_cache/mats.pkl')

if _path_metric_mats.exists(): 
    with _path_metric_mats.open('rb') as f: 
        METRIC_MATS = pickle.load(f)

def upper_tri(mat: np.ndarray) -> np.ndarray:
    i, j = np.triu_indices(len(mat), k=1)
    return mat[i, j]

rows = []
for metric_name, metric_mat in METRIC_MATS.items():
    if not (metric_name.startswith('ncd') or metric_name.startswith('jsd')): 
        metric_mat *= -1
    for uriel_name, uriel_mat in uriel_mats.items():
        m     = upper_tri(metric_mat)
        u     = upper_tri(uriel_mat)

        rho, _ = spearmanr(m, u)
        r,   _ = pearsonr(m, u)
        rows.append({
            'metric':       metric_name,
            'URIEL':        uriel_name,
            'spearman_rho': rho,
            'pearson_r':  r,
        })

df_long = pd.DataFrame(rows)

df_rho = df_long.pivot(index='metric', columns='URIEL', values='spearman_rho')
df_rho.columns = [f'{c} (ρ)'      for c in df_rho.columns]
df_r   = df_long.pivot(index='metric', columns='URIEL', values='pearson_r')
df_r.columns   = [f'{c} (r)' for c in df_r.columns]

ordered_cols = [col for u in uriel_mats for col in (f'{u} (ρ)', f'{u} (r)')]
df_corr = pd.concat([df_rho, df_r], axis=1)[ordered_cols]

print('Spearman ρ and Pearson r vs URIEL distance:')
display(df_corr.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1).format('{:.3f}'))

Spearman ρ and Pearson r vs URIEL distance:


,family (ρ),family (r),phonological (ρ),phonological (r),syntax (ρ),syntax (r)
metric,,,,,,
bpe_overlap_k-25_mff-1e-5,0.437,0.721,0.257,0.245,0.555,0.570
bpe_overlap_k-25_mff-1e-5_sky,0.582,0.778,0.223,0.197,0.697,0.670
bpe_overlap_k-25_mff-1e-7,0.434,0.709,0.283,0.270,0.541,0.550
bpe_overlap_k-25_mff-1e-7_sky,0.580,0.770,0.252,0.219,0.685,0.656
jacc_word,0.727,0.703,0.103,0.074,0.692,0.526
jsd_char_n=3,0.442,0.744,0.208,0.171,0.517,0.603
jsd_char_n=4,0.478,0.767,0.207,0.165,0.542,0.623
jsd_char_n=5,0.497,0.775,0.201,0.158,0.551,0.628
jsd_char_n=6,0.503,0.776,0.192,0.154,0.553,0.626


In [ ]:
METRIC_CLASSES = {
    'NCD':  METRICS_NCD,
    'JSD':  METRICS_JSD,
    'BPE':  METRICS_BPE,
    'JACC': METRICS_JACC,
}

pair_labels = [f'{LANG_CODES[i]}-{LANG_CODES[j]}'
               for i, j in zip(*np.triu_indices(N, k=1))]

GERMANIC = {'en', 'de', 'nl', 'da', 'no', 'sv'}
SLAVIC   = {'cs', 'sk', 'si', 'pl', 'hr', 'rs'}
family_colors = [
    'tab:blue' if {LANG_CODES[i], LANG_CODES[j]} <= GERMANIC
    else 'tab:red'  if {LANG_CODES[i], LANG_CODES[j]} <= SLAVIC
    else 'tab:grey'
    for i, j in zip(*np.triu_indices(N, k=1))
]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue', label='within Germanic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:red',  label='within Slavic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:grey', label='cross-family'),
]

_path_figs = Path('figs')
_path_figs.mkdir(exist_ok=True)

for class_name, metric_list in METRIC_CLASSES.items():
    for uriel_name, uriel_mat in uriel_mats.items():
        n_cols = len(metric_list)
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 4), squeeze=False)
        u = upper_tri(uriel_mat)

        for col, m in enumerate(metric_list):
            mv   = upper_tri(METRIC_MATS[m['label']])

            ax = axes[0, col]
            rho, _ = spearmanr(mv, u)
            ax.scatter(mv, u, c=family_colors, s=40, alpha=0.8, zorder=2)
            for lbl, mx, uy in zip(pair_labels, mv, u):
                ax.annotate(lbl, (mx, uy), fontsize=6, ha='left', va='bottom',
                            xytext=(3, 2), textcoords='offset points')
            ax.set_xlabel(m['label'], fontsize=8)
            ax.set_ylabel(f'URIEL {uriel_name}', fontsize=8)
            ax.set_title(f'ρ = {rho:.3f}')
            ax.grid(True, alpha=0.3)

        fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0))
        fig.suptitle(f'{class_name} vs URIEL {uriel_name}', fontsize=13)
        plt.tight_layout()
        fname = _path_figs / f'bible_uriel_{class_name}_{uriel_name}.pdf'
        plt.savefig(fname, dpi=150, bbox_inches='tight', format='pdf')
        plt.show()
        print(f'Saved {fname}')

## 5. Asymmetric size experiment

Simulates a realistic setting where a large reference corpus is compared against
a much smaller target corpus. For each reference language, the target is randomly
sampled to 1/R of the reference byte size, for **R ∈ {4, 8, 16}**.

The result for each ratio is an **asymmetric N×N matrix**: `mat[ref, tgt]` is the distance
from the full reference corpus to the small target sample.

Metrics evaluated: `jacc_word`, `jsd_char_n=5`, `ncd_{zlib,lzma,bz2}_65536`,
`bpe_overlap_k-25_mff-1e-5`, `bpe_overlap_k-25_mff-1e-5_sky`.

In [54]:
import random

ASYM_METRIC_LABELS = [
    'jacc_word',
    'jsd_char_n=5',
    'ncd_zlib_65536',
    'ncd_lzma_65536',
    'ncd_bz2_65536',
    'bpe_overlap_k-25_mff-1e-5_sky',
    'bpe_overlap_k-25_mff-1e-5',
]
ASYM_METRICS_SEL = sorted(
    [m for m in METRICS if m['label'] in ASYM_METRIC_LABELS],
    key=lambda m: ASYM_METRIC_LABELS.index(m['label']),
)

SIZE_RATIOS = [4, 8, 16]

ASYM_PKL = Path('_bible_cache/asym_mats_v2.pkl')
if ASYM_PKL.exists():
    with ASYM_PKL.open('rb') as f:
        ASYM_MATS = pickle.load(f)
    print('Loaded checkpoint.')
else:
    ASYM_MATS = {
        r: {m['label']: np.full((N, N), np.nan) for m in ASYM_METRICS_SEL}
        for r in SIZE_RATIOS
    }

def sample_to_bytes(docs: list, target_bytes: int, seed: int = 42) -> list:
    rng = random.Random(seed)
    order = list(range(len(docs)))
    rng.shuffle(order)
    sample, total = [], 0
    for idx in order:
        sample.append(docs[idx])
        total += len(docs[idx])
        if total >= target_bytes:
            break
    return sample

for ratio in SIZE_RATIOS:
    print(f'\n=== ratio 1/{ratio} ===')
    for i, ref_code in enumerate(LANG_CODES):
        ref_corpus  = corpora[ref_code]
        ref_bytes   = sum(len(d) for d in ref_corpus)
        target_bytes = ref_bytes // ratio
        print(f'  ref={ref_code} ({ref_bytes/1e6:.1f}MB) → sample={target_bytes/1e3:.0f}KB', flush=True)
        for j, tgt_code in enumerate(LANG_CODES):
            if i == j:
                continue
            if not any(np.isnan(ASYM_MATS[ratio][m['label']][i, j]) for m in ASYM_METRICS_SEL):
                continue
            tgt_sample = sample_to_bytes(corpora[tgt_code], target_bytes)
            for m in ASYM_METRICS_SEL:
                if not np.isnan(ASYM_MATS[ratio][m['label']][i, j]):
                    continue
                ASYM_MATS[ratio][m['label']][i, j] = m['fn'](ref_corpus, tgt_sample)
        with ASYM_PKL.open('wb') as f:
            pickle.dump(ASYM_MATS, f)

print('\nDone.')


=== ratio 1/4 ===
  ref=en (3.3MB) → sample=837KB
  ref=de (3.3MB) → sample=834KB
  ref=nl (3.3MB) → sample=834KB
  ref=da (2.9MB) → sample=725KB
  ref=no (3.0MB) → sample=753KB
  ref=sv (3.3MB) → sample=832KB
  ref=cs (2.9MB) → sample=728KB
  ref=sk (3.0MB) → sample=745KB
  ref=si (3.0MB) → sample=755KB
  ref=pl (3.2MB) → sample=791KB
  ref=hr (2.7MB) → sample=680KB
  ref=rs (2.6MB) → sample=653KB
  ref=ro (3.2MB) → sample=812KB
  ref=hu (3.2MB) → sample=806KB
  ref=fi (3.5MB) → sample=870KB

=== ratio 1/8 ===
  ref=en (3.3MB) → sample=419KB
  ref=de (3.3MB) → sample=417KB
  ref=nl (3.3MB) → sample=417KB
  ref=da (2.9MB) → sample=362KB
  ref=no (3.0MB) → sample=377KB
  ref=sv (3.3MB) → sample=416KB
  ref=cs (2.9MB) → sample=364KB
  ref=sk (3.0MB) → sample=372KB
  ref=si (3.0MB) → sample=377KB
  ref=pl (3.2MB) → sample=395KB
  ref=hr (2.7MB) → sample=340KB


: 

In [ ]:
# checkpoint is already saved per-ref inside the loop above;
# this cell re-saves the final state explicitly
with ASYM_PKL.open('wb') as f:
    pickle.dump(ASYM_MATS, f)
print(f'Saved {ASYM_PKL}')

In [ ]:
n_metrics = len(ASYM_METRICS_SEL)
ncols = 4
nrows = -(- n_metrics // ncols)  # ceiling division

for ratio in SIZE_RATIOS:
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows))
    axes = np.array(axes).flatten()
    for ax, m in zip(axes, ASYM_METRICS_SEL):
        mat = ASYM_MATS[ratio][m['label']]
        vmin, vmax = np.nanmin(mat), np.nanmax(mat)
        sns.heatmap(
            mat, ax=ax, annot=True, fmt='.3f', cmap='YlOrRd',
            xticklabels=LANG_CODES, yticklabels=LANG_CODES,
            vmin=vmin, vmax=vmax, linewidths=0.4,
            mask=np.eye(N, dtype=bool),
        )
        ax.set_title(m['label'], fontsize=9)
        ax.set_xlabel(f'target (1/{ratio})', fontsize=8)
        ax.set_ylabel('reference (full)', fontsize=8)
    for ax in axes[n_metrics:]:
        ax.set_visible(False)
    fig.suptitle(f'Asymmetric distances: full ref vs 1/{ratio} target', fontsize=13)
    plt.tight_layout()
    fname = f'bible_asym_heatmap_1-{ratio}.pdf'
    plt.savefig(fname, dpi=150, bbox_inches='tight', format='pdf')
    plt.show()
    print(f'Saved {fname}')

In [ ]:
off_diag = [(i, j) for i in range(N) for j in range(N) if i != j]

rows = []
for ratio in SIZE_RATIOS:
    for m in ASYM_METRICS_SEL:
        metric_vals = np.array([ASYM_MATS[ratio][m['label']][i, j] for i, j in off_diag])
        for uriel_name, uriel_mat in uriel_mats.items():
            uriel_vals = np.array([uriel_mat[i, j] for i, j in off_diag])
            rho, _ = spearmanr(metric_vals, uriel_vals)
            rows.append({'ratio': f'1/{ratio}', 'metric': m['label'], 'URIEL': uriel_name, 'rho': rho})

df_asym_corr = (
    pd.DataFrame(rows)
    .pivot(index=['ratio', 'metric'], columns='URIEL', values='rho')
)
print('Spearman ρ — asymmetric metrics vs URIEL (all off-diagonal pairs, both directions):')
display(df_asym_corr.style.background_gradient(cmap='RdYlGn', vmin=-1, vmax=1).format('{:.3f}'))

In [ ]:
dir_labels = [f'{LANG_CODES[i]}→{LANG_CODES[j]}' for i, j in off_diag]
dir_colors = [
    'tab:blue' if {LANG_CODES[i], LANG_CODES[j]} <= GERMANIC
    else 'tab:red'  if {LANG_CODES[i], LANG_CODES[j]} <= SLAVIC
    else 'tab:grey'
    for i, j in off_diag
]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:blue', label='within Germanic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:red',  label='within Slavic'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='tab:grey', label='cross-family'),
]

# one PDF per URIEL set: rows = ratios, cols = metrics
for uriel_name, uriel_mat in uriel_mats.items():
    u_all = np.array([uriel_mat[i, j] for i, j in off_diag])
    n_rows, n_cols = len(SIZE_RATIOS), len(ASYM_METRICS_SEL)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows), squeeze=False)

    for row, ratio in enumerate(SIZE_RATIOS):
        for col, m in enumerate(ASYM_METRICS_SEL):
            ax = axes[row, col]
            mv = np.array([ASYM_MATS[ratio][m['label']][i, j] for i, j in off_diag])
            rho, _ = spearmanr(mv, u_all)
            ax.scatter(mv, u_all, c=dir_colors, s=25, alpha=0.7, zorder=2)
            for lbl, x, y in zip(dir_labels, mv, u_all):
                ax.annotate(lbl, (x, y), fontsize=5, ha='left', va='bottom',
                            xytext=(2, 2), textcoords='offset points')
            ax.set_xlabel(m['label'] if row == n_rows - 1 else '', fontsize=7)
            ax.set_ylabel(f'URIEL {uriel_name}\n(1/{ratio})' if col == 0 else '', fontsize=7)
            ax.set_title(f'ρ = {rho:.3f}', fontsize=8)
            ax.grid(True, alpha=0.3)

    fig.legend(handles=legend_elements, loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=8)
    fig.suptitle(f'Asymmetric metrics vs URIEL {uriel_name}', fontsize=12)
    plt.tight_layout()
    fname = f'bible_asym_uriel_{uriel_name}.pdf'
    plt.savefig(fname, dpi=150, bbox_inches='tight', format='pdf')
    plt.show()
    print(f'Saved {fname}')